In [1]:
# ── Cell 1: Install required packages ──
!pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install --quiet timm scikit-learn pandas pillow tqdm



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# ── Cell 2: Imports, device & AMP scaler ──
import torch
from torch.cuda.amp import autocast, GradScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler()

print("Running on", DEVICE)


Running on cuda


/tmp/ipykernel_550/1231094034.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell: Download two ZIP archives from Dropbox, extract all images into images_final/
# ─────────────────────────────────────────────────────────────

!wget -q -O part1.zip "https://www.dropbox.com/scl/fi/qxevt9n8vr0usjf1csw9u/imagess.zip?rlkey=2bnzs6dzcvouf33uz4csyj2xq&st=6zdododq&dl=0"
!wget -q -O part2.zip "https://www.dropbox.com/scl/fi/r0fvnr7cwwv05s4clr1tw/images2.zip?rlkey=8n0y7p8fsdaevqtjhw29rlsxq&st=tc43357x&dl=0"

# 2) Unzip into temporary directories
!unzip -q part1.zip -d temp1
!unzip -q part2.zip -d temp2

# 3) Create the final images_final/ folder
!mkdir -p images_final

# 4) Move all common image files from both temp dirs into images_final/
!find temp1 temp2 -type f \( -iname '*.png' -o -iname '*.jpg' -o -iname '*.jpeg' -o -iname '*.dcm' \) -exec mv {} images_final/ \;

# 5) (Optional) Clean up temp folders and ZIP files
!rm -rf temp1 temp2 part1.zip part2.zip

# 6) Verify count
!echo "Collected $(ls images_final | wc -l) images into images_final/"


Collected 92120 images into images_final/


In [4]:
from pathlib import Path

# point to your folder
images_dir = Path("images_final")

# count all files in the directory
num_images = sum(1 for p in images_dir.iterdir() if p.is_file())

print(f"Number of images in 'images_final': {num_images}")


Number of images in 'images_final': 92120


In [7]:
# Final CNN training & evaluation using best hyperparameters

import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm.auto import tqdm
from torch.cuda.amp import autocast, GradScaler

# ── 0) Paths & device/scaler setup ──
IMAGES_DIR    = Path("images_final")
TEST_DIR      = Path("test_images_final")
CSV_PATH      = Path("Data_Entry_2017.csv")
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler        = GradScaler()

# ── 1) Load & preprocess labels ──
df = pd.read_csv(CSV_PATH)
df['Finding Labels'] = df['Finding Labels'].str.replace('No Finding','')
df['has_disease']    = (df['Finding Labels'] != '').astype(int)

# ── 2) Filter to images_final ──
final_names = {p.name for p in IMAGES_DIR.iterdir() if p.is_file()}
df_final    = df[df['Image Index'].isin(final_names)].reset_index(drop=True)
print(f"Found {len(df_final)} images in 'images_final/'")

# ── 3) Train/val split ──
train_df, val_df = train_test_split(
    df_final, 
    test_size=0.1, 
    stratify=df_final['has_disease'], 
    random_state=42
)
print(f"Train: {len(train_df)}  |  Val: {len(val_df)}")

# ── 4) Dataset & transforms ──
class CXR(Dataset):
    def __init__(self, df, root, tfm):
        self.df, self.root, self.tfm = df, root, tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.root / row['Image Index']).convert('L')
        img = self.tfm(img).repeat(3, 1, 1)
        return img, torch.tensor([row['has_disease']], dtype=torch.float32)

tfm = {
    'train': T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomRotation(10),
        T.RandomResizedCrop(224, scale=(0.8,1.0)),
        T.ToTensor(), T.Normalize([0.485],[0.229])
    ]),
    'val':   T.Compose([
        T.Resize(240), T.CenterCrop(224),
        T.ToTensor(), T.Normalize([0.485],[0.229])
    ])
}

train_dl = DataLoader(CXR(train_df, IMAGES_DIR, tfm['train']),
                      batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(CXR(val_df,   IMAGES_DIR, tfm['val']),
                      batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

# ── 5) Build model with best cfg ──
backbone = models.densenet121(pretrained=True)
feat_dim = backbone.classifier.in_features
backbone.classifier = nn.Identity()
model = nn.Sequential(
    backbone,
    nn.Dropout(0.5),
    nn.Linear(feat_dim, 1)
).to(DEVICE)

# compute pos_weight for imbalance
counts = train_df['has_disease'].value_counts().to_dict()
pos_weight = torch.tensor(counts[0] / counts[1]).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=7)

# ── 6) Training loop (7 epochs) ──
best_auc = 0.0
for ep in range(1, 8):
    print(f"\n=== Epoch {ep}/7 ===")
    # train
    model.train()
    tot_loss = 0.0
    for xb, yb in tqdm(train_dl, desc="Train"):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        with autocast():
            out  = model(xb)
            loss = criterion(out, yb)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        tot_loss += loss.item() * xb.size(0)
    print(f"Train loss: {tot_loss/len(train_df):.4f}")

    # validate
    model.eval()
    preds, targs = [], []
    with torch.no_grad():
        for xb, yb in tqdm(val_dl, desc="Val"):
            xb = xb.to(DEVICE)
            out = model(xb)
            preds.append(torch.sigmoid(out).cpu())
            targs.append(yb)
    preds = torch.cat(preds).squeeze().numpy()
    targs = torch.cat(targs).squeeze().numpy()
    val_auc = roc_auc_score(targs, preds)
    print(f"Val ROC-AUC: {val_auc:.4f}")
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), 'final_best_model.pth')
        print("✔ New best model saved")
    scheduler.step()

print(f"\nBest validation AUC: {best_auc:.4f}")



/tmp/ipykernel_550/354120326.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler        = GradScaler()
/opt/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Found 92120 images in 'images_final/'
Train: 82908  |  Val: 9212

=== Epoch 1/7 ===


Train:   0%|          | 0/5182 [00:00<?, ?it/s]/tmp/ipykernel_550/354120326.py:97: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train: 100%|██████████| 5182/5182 [06:41<00:00, 12.90it/s]


Train loss: 0.6707


Val: 100%|██████████| 576/576 [00:37<00:00, 15.51it/s]


...

Val ROC-AUC: 0.7803
✔ New best model saved

=== Epoch 7/7 ===


Train:   0%|          | 0/5182 [00:00<?, ?it/s]/tmp/ipykernel_550/354120326.py:97: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Train: 100%|██████████| 5182/5182 [06:37<00:00, 13.03it/s]


Train loss: 0.5960


Val: 100%|██████████| 576/576 [00:37<00:00, 15.52it/s]


Val ROC-AUC: 0.7817
✔ New best model saved

Best validation AUC: 0.7817


In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell: Evaluate on your 10 000-image test set with full metrics
# ─────────────────────────────────────────────────────────────

import numpy as np
import torch
from pathlib import Path
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T, models
from torch import nn
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# 1) Paths and device
TEST_DIR   = Path("test_images_final")
CSV_PATH   = Path("Data_Entry_2017.csv")
WEIGHTS    = "final_best_model.pth"       # your best model
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2) Load labels
df = pd.read_csv(CSV_PATH)
df['Finding Labels'] = df['Finding Labels'].str.replace('No Finding','')
df['has_disease']    = (df['Finding Labels'] != '').astype(int)

# 3) Filter to only those in test folder
test_names = {p.name for p in TEST_DIR.iterdir() if p.is_file()}
test_df    = df[df['Image Index'].isin(test_names)].reset_index(drop=True)
print(f"→ Found {len(test_df)} images for testing\n")

# 4) Preprocessing pipeline (must match validation)
tfm_val = T.Compose([
    T.Resize(240),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485], [0.229])
])

class CXRTest(Dataset):
    def __init__(self, data, root, tfm):
        self.df, self.root, self.tfm = data, root, tfm
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.root/row['Image Index']).convert('L')
        img = self.tfm(img).repeat(3,1,1)
        return img, row['has_disease']

test_dl = DataLoader(
    CXRTest(test_df, TEST_DIR, tfm_val),
    batch_size=32, shuffle=False, num_workers=2, pin_memory=True
)

# 5) Rebuild exactly the same model you trained
backbone = models.densenet121(pretrained=False)
feat_dim = backbone.classifier.in_features
backbone.classifier = nn.Identity()
model = nn.Sequential(
    backbone,
    nn.Dropout(0.5),
    nn.Linear(feat_dim, 1)
).to(DEVICE)

# 6) Load weights
model.load_state_dict(torch.load(WEIGHTS, map_location=DEVICE))
model.eval()

# 7) Inference
probs, labels = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        xb = xb.to(DEVICE)
        out = model(xb)
        p = torch.sigmoid(out).cpu().numpy().flatten()
        probs.extend(p.tolist())
        labels.extend(yb.tolist())

probs  = np.array(probs)
labels = np.array(labels)
preds  = (probs >= 0.5).astype(int)

# 8) Metrics
print("Classification report:")
print(classification_report(labels, preds, digits=4))

cm = confusion_matrix(labels, preds)
print("Confusion matrix:")
print(cm)

print(f"Accuracy : {accuracy_score(labels, preds):.4f}")
print(f"ROC-AUC  : {roc_auc_score(labels, probs):.4f}")
print(f"Precision: {precision_score(labels, preds):.4f}")
print(f"Recall   : {recall_score(labels, preds):.4f}")
print(f"F1 score : {f1_score(labels, preds):.4f}")


In [8]:
# 1) Install gdown (if not already)
!pip install -q gdown

# 2) Download via the full share URL, letting gdown find the file ID
!gdown --fuzzy "https://drive.google.com/file/d/12dUqEty10Tc6S4btnqWfeUyhTc4VqfeL/view?usp=sharing" \
        -O images1.zip

# 3) Create the folder to receive the images
!mkdir -p images1

# 4) Unzip into that folder
!unzip -q images1.zip -d test_images

# 5) Clean up the ZIP
!rm images1.zip

# 6) Confirm file count
!echo "Extracted $(ls images1 | wc -l) files into test_images/"



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Downloading...
From (original): https://drive.google.com/uc?id=12dUqEty10Tc6S4btnqWfeUyhTc4VqfeL
From (redirected): https://drive.google.com/uc?id=12dUqEty10Tc6S4btnqWfeUyhTc4VqfeL&confirm=t&uuid=382361ca-bbd6-48e4-b890-0f0b6d05f340
To: /workspace/images1.zip
100%|██████████████████████████████████████| 4.06G/4.06G [02:54<00:00, 23.3MB/s]
Extracted 0 files into test_images/


In [9]:
# ─────────────────────────────────────────────────────────────
# Cell: Evaluate on your 10 000-image test set with full metrics
# ─────────────────────────────────────────────────────────────

import numpy as np
import torch
from pathlib import Path
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T, models
from torch import nn
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# 1) Paths and device
TEST_DIR   = Path("test_images") / "images1"    # ← now pointing inside images1
CSV_PATH   = Path("Data_Entry_2017.csv")
WEIGHTS    = "final_best_model.pth"
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2) Load & preprocess labels
df = pd.read_csv(CSV_PATH)
df['Finding Labels'] = df['Finding Labels'].str.replace('No Finding','')
df['has_disease']    = (df['Finding Labels'] != '').astype(int)

# 3) Filter to only those in test_images/images1/
test_names = {p.name for p in TEST_DIR.iterdir() if p.is_file()}
test_df    = df[df['Image Index'].isin(test_names)].reset_index(drop=True)
print(f"→ Found {len(test_df)} images for testing in '{TEST_DIR}/'\n")

# 4) Preprocessing pipeline
tfm_val = T.Compose([
    T.Resize(240),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485], [0.229])
])

class CXRTest(Dataset):
    def __init__(self, df, root, tfm):
        self.df, self.root, self.tfm = df, root, tfm
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.root/row['Image Index']).convert('L')
        img = self.tfm(img).repeat(3,1,1)
        return img, row['has_disease']

test_dl = DataLoader(
    CXRTest(test_df, TEST_DIR, tfm_val),
    batch_size=32, shuffle=False, num_workers=2, pin_memory=True
)

# 5) Rebuild your trained model architecture
backbone = models.densenet121(pretrained=False)
feat_dim = backbone.classifier.in_features
backbone.classifier = nn.Identity()
model = nn.Sequential(
    backbone,
    nn.Dropout(0.5),
    nn.Linear(feat_dim, 1)
).to(DEVICE)

# 6) Load saved weights
model.load_state_dict(torch.load(WEIGHTS, map_location=DEVICE))
model.eval()

# 7) Inference
probs, labels = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        xb = xb.to(DEVICE)
        out = model(xb)
        p = torch.sigmoid(out).cpu().numpy().flatten()
        probs.extend(p.tolist())
        labels.extend(yb.tolist())

probs  = np.array(probs)
labels = np.array(labels)
preds  = (probs >= 0.5).astype(int)

# 8) Compute & print metrics
print("Classification report:")
print(classification_report(labels, preds, digits=4))

cm = confusion_matrix(labels, preds)
print("Confusion matrix:")
print(cm)

print(f"Accuracy : {accuracy_score(labels, preds):.4f}")
print(f"ROC-AUC  : {roc_auc_score(labels, probs):.4f}")
print(f"Precision: {precision_score(labels, preds):.4f}")
print(f"Recall   : {recall_score(labels, preds):.4f}")
print(f"F1 score : {f1_score(labels, preds):.4f}")


→ Found 10000 images for testing in 'test_images/images1/'



/opt/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Classification report:
              precision    recall  f1-score   support

           0     0.7596    0.7371    0.7482      5375
           1     0.7046    0.7289    0.7165      4625

    accuracy                         0.7333     10000
   macro avg     0.7321    0.7330    0.7324     10000
weighted avg     0.7342    0.7333    0.7336     10000

Confusion matrix:
[[3962 1413]
 [1254 3371]]
Accuracy : 0.7333
ROC-AUC  : 0.8004
Precision: 0.7046
Recall   : 0.7289
F1 score : 0.7165
